# Exercises: Classes and Numpy

**Table of contents**<a id='toc0_'></a>    
- 1. [A firm as a class](#toc1_)    
- 2. [Adding a solve method](#toc2_)    
- 3. [Flexible parameters with `**kwargs`](#toc3_)    
- 4. [Operator methods: adding bundles](#toc4_)    
- 5. [Creating arrays](#toc5_)    
- 6. [View vs. copy](#toc6_)    
- 7. [Element-by-element vs. matrix products](#toc7_)    
- 8. [Broadcasting: a utility grid](#toc8_)    
- 9. [Logical indexing](#toc9_)    
- 10. [Reducing along an axis](#toc10_)    
- 11. [Putting it together: solving a consumer problem on a grid](#toc11_)    
- 12. [Summary](#toc12_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

These exercises cover the notebooks *Classes* and *Numpy basics*. Exercises 1-4 are about classes, 5-10 about numpy, and exercise 11 combines the two.

**How to work:** write your own solution in the empty cell below each task, run it, and *only then* look at the answer.

Most answers end with a **check**: the result is compared to something we already know is true - an analytical formula, an equivalent calculation, or a slow-but-obviously-correct loop. Vectorized numpy code is easy to get subtly wrong, so this habit matters even more here than in the previous lecture.

In [1]:
import numpy as np

## 1. <a id='toc1_'></a>[A firm as a class](#toc0_)

**Task:** A firm produces with the technology $y = A\ell^{\gamma}$ and sells at price $p$, paying the wage $w$ per unit of labor. Profits are

$$
\pi(\ell) = pA\ell^{\gamma} - w\ell
$$

Write a class `Firm` with

1. an `__init__` method storing `A`, `gamma`, `w` and `p` as attributes (use the default values $A=2$, $\gamma=0.5$, $w=1$, $p=1$),
2. a method `production(ell)` returning output,
3. a method `profit(ell)` returning profits.

Create a firm with the default parameters and print output and profits at $\ell = 4$.

In [2]:
# write your code here

**Answer:**

In [3]:
class Firm:

    def __init__(self,A=2.0,gamma=0.5,w=1.0,p=1.0):
        """Store the parameters of the firm as attributes."""

        self.A = A
        self.gamma = gamma
        self.w = w
        self.p = p

    def production(self,ell):
        """Output produced with ell units of labor."""

        return self.A*ell**self.gamma

    def profit(self,ell):
        """Profits from using ell units of labor."""

        return self.p*self.production(ell) - self.w*ell

firm = Firm()
print('output at ell = 4:', firm.production(4.0))
print('profit at ell = 4:', firm.profit(4.0))

output at ell = 4: 4.0
profit at ell = 4: 0.0


**Check:** with $A=2$, $\gamma=0.5$, $\ell=4$ we have $y = 2\sqrt{4}$, and with $p=w=1$ profits are $y-\ell$.

In [4]:
import math

assert math.isclose(firm.production(4.0), 2.0*math.sqrt(4.0))
assert math.isclose(firm.profit(4.0), 2.0*math.sqrt(4.0)-4.0)
print('check passed')

check passed


**Note:** `profit` calls `self.production(ell)` instead of repeating the formula. The parameters do not have to be passed around: `self` gives every method access to all attributes.

## 2. <a id='toc2_'></a>[Adding a solve method](#toc0_)

**Task:** The first order condition $p A\gamma\ell^{\gamma-1} = w$ gives the optimal labor input

$$
\ell^{\ast} = \left(\frac{\gamma pA}{w}\right)^{\frac{1}{1-\gamma}}
$$

Extend the class with

1. a method `solve()` that computes $\ell^{\ast}$ and stores it, together with the resulting output and profits, as attributes,
2. a `__str__` method so that `print(firm)` gives a readable summary.

Then solve the firm's problem with the default parameters.

In [5]:
# write your code here

**Answer:**

In [6]:
class Firm:

    def __init__(self,A=2.0,gamma=0.5,w=1.0,p=1.0):
        """Store the parameters of the firm as attributes."""

        self.A = A
        self.gamma = gamma
        self.w = w
        self.p = p

        self.ell_star = None # not solved yet
        self.y_star = None
        self.profit_star = None

    def __str__(self):
        """Text summary of parameters and solution."""

        text = f'Firm(A={self.A}, gamma={self.gamma}, w={self.w}, p={self.p})\n'
        if self.ell_star is None:
            text += ' not solved yet'
        else:
            text += f' ell* = {self.ell_star:.4f}\n'
            text += f' y*   = {self.y_star:.4f}\n'
            text += f' pi*  = {self.profit_star:.4f}'

        return text

    def production(self,ell):
        """Output produced with ell units of labor."""

        return self.A*ell**self.gamma

    def profit(self,ell):
        """Profits from using ell units of labor."""

        return self.p*self.production(ell) - self.w*ell

    def solve(self):
        """Solve for optimal labor input and store the solution as attributes."""

        self.ell_star = (self.gamma*self.p*self.A/self.w)**(1/(1-self.gamma))
        self.y_star = self.production(self.ell_star)
        self.profit_star = self.profit(self.ell_star)

firm = Firm()
print(firm)
print('')

firm.solve()
print(firm)

Firm(A=2.0, gamma=0.5, w=1.0, p=1.0)
 not solved yet

Firm(A=2.0, gamma=0.5, w=1.0, p=1.0)
 ell* = 1.0000
 y*   = 2.0000
 pi*  = 1.0000


**Check:** at the optimum, profits must be higher than at any nearby labor input. Instead of trusting the formula, we simply *try* some neighboring values.

In [7]:
pi_star = firm.profit(firm.ell_star)
for step in [-0.5,-0.1,-0.01,0.01,0.1,0.5]:
    pi = firm.profit(firm.ell_star+step)
    print(f'step = {step:6.2f}: profit = {pi:.8f} (lower than pi*: {pi < pi_star})')
    assert pi < pi_star

print('check passed: the solution is a local maximum')

step =  -0.50: profit = 0.91421356 (lower than pi*: True)
step =  -0.10: profit = 0.99736660 (lower than pi*: True)
step =  -0.01: profit = 0.99997487 (lower than pi*: True)
step =   0.01: profit = 0.99997512 (lower than pi*: True)
step =   0.10: profit = 0.99761770 (lower than pi*: True)
step =   0.50: profit = 0.94948974 (lower than pi*: True)
check passed: the solution is a local maximum


## 3. <a id='toc3_'></a>[Flexible parameters with `**kwargs`](#toc0_)

**Task:** Copy your `Firm` class and change `__init__` so that it takes `**kwargs` instead of named arguments: set all parameters to their default values inside `__init__`, and then overwrite the ones given as keyword arguments (use `setattr`, as in the `Agent` class from the lecture).

Then create three firms with wages $w \in \{0.5, 1.0, 2.0\}$, solve them, and print the optimal labor input for each. Does labor demand fall with the wage?

In [8]:
# write your code here

**Answer:**

In [9]:
class Firm:

    def __init__(self,**kwargs):
        """Set default parameters and overwrite them with any keyword arguments."""

        # a. defaults
        self.A = 2.0
        self.gamma = 0.5
        self.w = 1.0
        self.p = 1.0

        self.ell_star = None
        self.y_star = None
        self.profit_star = None

        # b. overwrite
        for key,value in kwargs.items():
            setattr(self,key,value) # like self.key = value

    def production(self,ell):
        """Output produced with ell units of labor."""

        return self.A*ell**self.gamma

    def profit(self,ell):
        """Profits from using ell units of labor."""

        return self.p*self.production(ell) - self.w*ell

    def solve(self):
        """Solve for optimal labor input and store the solution as attributes."""

        self.ell_star = (self.gamma*self.p*self.A/self.w)**(1/(1-self.gamma))
        self.y_star = self.production(self.ell_star)
        self.profit_star = self.profit(self.ell_star)

firms = []
for w in [0.5,1.0,2.0]:

    firm = Firm(w=w)
    firm.solve()
    firms.append(firm)

    print(f'w = {w:.1f}: ell* = {firm.ell_star:.4f}, y* = {firm.y_star:.4f}, pi* = {firm.profit_star:.4f}')

w = 0.5: ell* = 4.0000, y* = 4.0000, pi* = 2.0000
w = 1.0: ell* = 1.0000, y* = 2.0000, pi* = 1.0000
w = 2.0: ell* = 0.2500, y* = 1.0000, pi* = 0.5000


**Check:** labor demand must be strictly decreasing in the wage.

In [10]:
ell_stars = [firm.ell_star for firm in firms]
print('ell* =', ell_stars)
assert ell_stars[0] > ell_stars[1] > ell_stars[2]
print('check passed: labor demand is decreasing in the wage')

ell* = [4.0, 1.0, 0.25]
check passed: labor demand is decreasing in the wage


**Note:** the three firms are three *separate objects*. Changing `firms[0].w` does not affect `firms[1]`. This is the whole point of using a class: you can create as many parameterizations as you want and solve them independently.

## 4. <a id='toc4_'></a>[Operator methods: adding bundles](#toc0_)

**Task:** Write a class `Bundle` describing a consumption bundle of two goods, $(x_1,x_2)$, with

1. `__init__(self,x1,x2)`,
2. `__str__` so that `print` shows something like `(2.0, 3.0)`,
3. `__add__` so that adding two bundles adds them good by good,
4. `__mul__` so that multiplying a bundle by a *number* scales both goods.

Test it by adding two bundles and doubling the result.

**Hint:** `__mul__(self,other)` is called for `bundle*2`, and here `other` is a number, not a `Bundle`.

In [11]:
# write your code here

**Answer:**

In [12]:
class Bundle:

    def __init__(self,x1,x2):
        """A consumption bundle of two goods."""

        self.x1 = x1
        self.x2 = x2

    def __str__(self): # called by print
        return f'({self.x1}, {self.x2})'

    def __add__(self,other): # called by +
        return Bundle(self.x1+other.x1, self.x2+other.x2)

    def __mul__(self,factor): # called by *
        return Bundle(self.x1*factor, self.x2*factor)

a = Bundle(2.0,3.0)
b = Bundle(1.0,5.0)

c = a+b
print('a =',a)
print('b =',b)
print('a+b =',c)
print('(a+b)*2 =',c*2)

a = (2.0, 3.0)
b = (1.0, 5.0)
a+b = (3.0, 8.0)
(a+b)*2 = (6.0, 16.0)


**Check:** compare with the numbers computed directly.

In [13]:
assert c.x1 == 2.0+1.0 and c.x2 == 3.0+5.0
d = c*2
assert d.x1 == 2*c.x1 and d.x2 == 2*c.x2
print('check passed')

check passed


**Note:** `a+b` returns a **new** `Bundle` and leaves `a` and `b` untouched - exactly like `1+2` does not change `1`. Also note that `2*c` (number first) does *not* work: that would require the method `__rmul__`. Try it and read the error message.

## 5. <a id='toc5_'></a>[Creating arrays](#toc0_)

**Task:**

1. Create a numpy array `x` with 11 equally spaced points from 0 to 1 (both endpoints included).
2. Create a $3\times 4$ array of zeros and a $2\times 2$ identity matrix.
3. Print `ndim`, `shape`, `size` and `dtype` for `x` and for the $3\times 4$ array.
4. Reshape `np.arange(6)` into a $2\times 3$ array and print it.

In [14]:
# write your code here

**Answer:**

In [15]:
# 1.
x = np.linspace(0,1,11)
print('x =',x)

# 2.
Z = np.zeros((3,4))
I = np.eye(2)
print('Z =\n',Z)
print('I =\n',I)

# 3.
print('x: ndim =',x.ndim,'shape =',x.shape,'size =',x.size,'dtype =',x.dtype)
print('Z: ndim =',Z.ndim,'shape =',Z.shape,'size =',Z.size,'dtype =',Z.dtype)

# 4.
B = np.arange(6).reshape(2,3)
print('B =\n',B)
print('B dtype =',B.dtype) # integers, because arange was given integers

x = [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
Z =
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
I =
 [[1. 0.]
 [0. 1.]]
x: ndim = 1 shape = (11,) size = 11 dtype = float64
Z: ndim = 2 shape = (3, 4) size = 12 dtype = float64
B =
 [[0 1 2]
 [3 4 5]]
B dtype = int64


**Check:** the spacing in `x` should be 0.1 everywhere. Note that we use `np.allclose` and not `==`.

In [16]:
print('differences:',np.diff(x))
print('exactly 0.1 everywhere:',np.all(np.diff(x) == 0.1)) # floating point!
print('close to 0.1 everywhere:',np.allclose(np.diff(x),0.1))

differences: [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
exactly 0.1 everywhere: False
close to 0.1 everywhere: True


**Note:** `np.linspace(0,1,11)` includes *both* endpoints and gives 11 points, while `np.arange(0,1,0.1)` excludes the endpoint and gives 10. Mixing the two up is a classic off-by-one error.

## 6. <a id='toc6_'></a>[View vs. copy](#toc0_)

**Task:** Consider the code below. **Predict what `A` is** after the last line before you run it. Then run it, explain the result, and fix it so that `A` is left unchanged.

In [17]:
A = np.array([1.0,2.0,3.0,4.0,5.0])

B = A[2:]  # the last three elements
B[:] = 0.0 # set them to zero

print('B =',B)
print('A =',A)

B = [0. 0. 0.]
A = [1. 2. 0. 0. 0.]


In [18]:
# write your fix here

**Answer:**

`A` is changed too. Slicing a numpy array returns a **view**, i.e. a reference into the memory of the original array - *not* a copy. This differs from a list, where a slice *is* a copy.

In [19]:
# lists and arrays behave differently!
A_list = [1.0,2.0,3.0,4.0,5.0]
B_list = A_list[2:]
B_list[0] = 0.0
print('list: A_list =',A_list,'-> unchanged')

A = np.array([1.0,2.0,3.0,4.0,5.0])
B = A[2:]
B[0] = 0.0
print('array: A =',A,'-> changed')

list: A_list = [1.0, 2.0, 3.0, 4.0, 5.0] -> unchanged
array: A = [1. 2. 0. 4. 5.] -> changed


The fix is to ask for a copy explicitly:

In [20]:
A = np.array([1.0,2.0,3.0,4.0,5.0])
B = A[2:].copy() # now an independent array
B[:] = 0.0

print('B =',B)
print('A =',A)

assert np.allclose(A,np.array([1.0,2.0,3.0,4.0,5.0]))
print('check passed: A is untouched')

B = [0. 0. 0.]
A = [1. 2. 3. 4. 5.]
check passed: A is untouched


**Note:** views are a *feature*, not a bug - they make numpy fast because no data is copied. But you have to know when you are working on a view. `B = A[2:]` gives a view; `B = A[2:].copy()` gives a copy.

## 7. <a id='toc7_'></a>[Element-by-element vs. matrix products](#toc0_)

**Task:** Let

$$
A = \begin{bmatrix} 1 & 2 \\ 3 & 4\end{bmatrix}, \qquad
B = \begin{bmatrix} 0 & 1 \\ 1 & 1\end{bmatrix}
$$

1. Compute `A*B` and `A@B` and explain the difference.
2. Compute the matrix product **by hand** (on paper) and check that `A@B` matches.
3. Check whether `A@B` and `B@A` are equal.

In [21]:
# write your code here

**Answer:**

In [22]:
A = np.array([[1.0,2.0],
              [3.0,4.0]])

B = np.array([[0.0,1.0],
              [1.0,1.0]])

print('A*B (element-by-element) =\n',A*B)
print('A@B (matrix product) =\n',A@B)
print('B@A (matrix product) =\n',B@A)

A*B (element-by-element) =
 [[0. 2.]
 [3. 4.]]
A@B (matrix product) =
 [[2. 3.]
 [4. 7.]]
B@A (matrix product) =
 [[3. 4.]
 [4. 6.]]


**Check:** the matrix product computed with an explicit double loop must give the same as `@`.

In [23]:
C = np.zeros((2,2))
for i in range(2):
    for j in range(2):
        for k in range(2):
            C[i,j] += A[i,k]*B[k,j]

print('loop =\n',C)
print('same as A@B:',np.allclose(C,A@B))

print('A@B equals B@A:',np.allclose(A@B,B@A)) # matrix multiplication is not commutative

loop =
 [[2. 3.]
 [4. 7.]]
same as A@B: True
A@B equals B@A: False


**Note:** `*` is *always* element-by-element in numpy, also for two-dimensional arrays. If you translate a matrix formula from a paper into code with `*`, you will silently get the wrong numbers. Use `@`.

## 8. <a id='toc8_'></a>[Broadcasting: a utility grid](#toc0_)

**Task:** Let `x1 = np.linspace(1,4,4)` and `x2 = np.linspace(1,3,3)`, and let $\alpha = 0.5$.

Compute a matrix `u` where `u[i,j]` is the Cobb-Douglas utility $x_{1,i}^{\alpha}x_{2,j}^{1-\alpha}$ of the bundle $(x_{1,i},x_{2,j})$ - **without writing a loop**. What is the shape of `u`?

**Hint:** use `None` (`np.newaxis`) to turn `x1` into a column and `x2` into a row.

In [24]:
# write your code here

**Answer:**

In [25]:
x1 = np.linspace(1,4,4)
x2 = np.linspace(1,3,3)
alpha = 0.5

print('x1:',x1,x1.shape)
print('x2:',x2,x2.shape)

u = x1[:,None]**alpha * x2[None,:]**(1-alpha) # (4,1) and (1,3) -> (4,3)

print('shape of u:',u.shape)
print('u =\n',u.round(4))

x1: [1. 2. 3. 4.] (4,)
x2: [1. 2. 3.] (3,)
shape of u: (4, 3)
u =
 [[1.     1.4142 1.7321]
 [1.4142 2.     2.4495]
 [1.7321 2.4495 3.    ]
 [2.     2.8284 3.4641]]


**Check:** the same thing with a slow double loop. This is the standard way of testing vectorized code - write the obvious version first, then vectorize, then compare.

In [26]:
u_loop = np.zeros((x1.size,x2.size))
for i in range(x1.size):
    for j in range(x2.size):
        u_loop[i,j] = x1[i]**alpha * x2[j]**(1-alpha)

print('identical:',np.allclose(u,u_loop))

# and one element against a hand calculation
print('u[3,2] =',u[3,2],'and 4**0.5 * 3**0.5 =',4**0.5*3**0.5)
assert np.isclose(u[3,2], 4**0.5*3**0.5)
print('check passed')

identical: True
u[3,2] = 3.4641016151377544 and 4**0.5 * 3**0.5 = 3.4641016151377544
check passed


**Note:** without `None` the multiplication `x1**alpha * x2**(1-alpha)` fails, because shapes `(4,)` and `(3,)` cannot be broadcast together. Try it and read the error message - "operands could not be broadcast together" almost always means you forgot a `None`.

## 9. <a id='toc9_'></a>[Logical indexing](#toc0_)

**Task:** The array below contains the yearly income (in 1000 DKK) of 8 individuals.

1. Compute the mean income.
2. Create a boolean array `I` which is `True` for individuals with an income above the mean.
3. Print how many individuals are above the mean, their share of the population, and their average income.
4. Print the incomes of those who earn *both* above 200 **and** below 400.

In [27]:
income = np.array([180.0,240.0,320.0,150.0,410.0,260.0,95.0,510.0])

In [28]:
# write your code here

**Answer:**

In [29]:
income = np.array([180.0,240.0,320.0,150.0,410.0,260.0,95.0,510.0])

# 1.
mean = np.mean(income)
print('mean income =',mean)

# 2.
I = income > mean
print('I =',I)

# 3.
print('number above mean =',np.sum(I))    # True counts as 1
print('share above mean =',np.mean(I))    # the mean of a boolean array is a share
print('their average income =',np.mean(income[I]))

# 4.
J = (income > 200) & (income < 400)       # note & and not 'and'
print('incomes between 200 and 400:',income[J])

mean income = 270.625
I = [False False  True False  True False False  True]
number above mean = 3
share above mean = 0.375
their average income = 413.3333333333333
incomes between 200 and 400: [240. 320. 260.]


**Check:** the population mean must be a weighted average of the mean below and the mean above.

In [30]:
share = np.mean(I)
mean_above = np.mean(income[I])
mean_below = np.mean(income[~I]) # ~ is 'not'

print('weighted average =',share*mean_above + (1-share)*mean_below)
print('mean             =',mean)
assert np.isclose(share*mean_above + (1-share)*mean_below, mean)
print('check passed')

weighted average = 270.625
mean             = 270.625
check passed


**Note:** use `&`, `|` and `~` for element-by-element logic on arrays. The Python keywords `and`, `or` and `not` raise an error on arrays with more than one element, because they try to convert the whole array to a single `True`/`False`.

## 10. <a id='toc10_'></a>[Reducing along an axis](#toc0_)

**Task:** The matrix below contains the returns (in pct.) of 3 assets (rows) over 4 years (columns).

1. Compute the average return **per asset**.
2. Compute the average return **per year**.
3. Compute the standard deviation per asset, and find the index of the asset with the highest average return using `argmax`.

**Hint:** `axis=0` reduces *down the rows*, `axis=1` reduces *across the columns*.

In [31]:
returns = np.array([[ 5.0, 7.0, 6.0, 2.0],
                    [12.0,-4.0, 9.0,-1.0],
                    [ 3.0, 3.0, 4.0, 2.0]])

In [32]:
# write your code here

**Answer:**

In [33]:
returns = np.array([[ 5.0, 7.0, 6.0, 2.0],
                    [12.0,-4.0, 9.0,-1.0],
                    [ 3.0, 3.0, 4.0, 2.0]])

print('shape:',returns.shape,'(assets,years)')

mean_asset = np.mean(returns,axis=1) # average across years, one number per asset
mean_year = np.mean(returns,axis=0)  # average across assets, one number per year

print('mean per asset:',mean_asset,mean_asset.shape)
print('mean per year :',mean_year,mean_year.shape)

std_asset = np.std(returns,axis=1)
print('std per asset :',std_asset.round(4))

best = np.argmax(mean_asset)
print('asset with highest average return: index',best,'with',mean_asset[best])

shape: (3, 4) (assets,years)
mean per asset: [5. 4. 3.] (3,)
mean per year : [6.66666667 2.         6.33333333 1.        ] (4,)
std per asset : [1.8708 6.6708 0.7071]
asset with highest average return: index 0 with 5.0


**Check:** because all assets are observed in all years, the mean of the asset means and the mean of the year means must both equal the overall mean.

In [34]:
print('overall mean       :',np.mean(returns))
print('mean of asset means:',np.mean(mean_asset))
print('mean of year means :',np.mean(mean_year))

assert np.isclose(np.mean(mean_asset),np.mean(returns))
assert np.isclose(np.mean(mean_year),np.mean(returns))
print('check passed')

overall mean       : 4.0
mean of asset means: 4.0
mean of year means : 4.0
check passed


**Note:** the safest way to remember the `axis` argument is to look at the **shape**: `returns` has shape `(3,4)`, so `axis=1` removes the second dimension and leaves shape `(3,)` - one number per asset. If the shape of your result is not what you expected, you used the wrong axis.

## 11. <a id='toc11_'></a>[Putting it together: solving a consumer problem on a grid](#toc0_)

**Task:** Consider the consumer problem

$$
\max_{x_1,x_2} x_1^{\alpha}x_2^{1-\alpha} \quad \text{s.t.} \quad p_1x_1+p_2x_2 = I
$$

with $\alpha = 0.5$, $I = 10$, $p_1 = 1$ and $p_2 = 2$. We know the solution analytically: $x_1^{\ast} = \alpha I/p_1$ and $x_2^{\ast} = (1-\alpha)I/p_2$.

Solve it **numerically** instead:

1. Write a class `ConsumerProblem` with the parameters as attributes.
2. Give it a method `solve(N)` that constructs a grid of `N` values of $x_1$ between 0 and $I/p_1$, computes $x_2 = (I-p_1x_1)/p_2$ for each of them (so the budget always binds), evaluates utility over the whole grid **without a loop**, and picks the best point with `argmax`.
3. Compare the numerical solution to the analytical one for `N = 11`, `N = 101` and `N = 1001`.

In [35]:
# write your code here

**Answer:**

In [36]:
class ConsumerProblem:

    def __init__(self,alpha=0.5,I=10.0,p1=1.0,p2=2.0):
        """Store the parameters of the consumer problem."""

        self.alpha = alpha
        self.I = I
        self.p1 = p1
        self.p2 = p2

    def u_func(self,x1,x2):
        """Cobb-Douglas utility (works for both scalars and arrays)."""

        return x1**self.alpha * x2**(1-self.alpha)

    def solve_analytical(self):
        """The known solution."""

        x1 = self.alpha*self.I/self.p1
        x2 = (1-self.alpha)*self.I/self.p2

        return x1,x2

    def solve(self,N):
        """Solve on a grid of N points for x1.

        Args:
            N (int): number of grid points

        Returns:
            x1 (float): best consumption of good 1 on the grid
            x2 (float): corresponding consumption of good 2
            u (float): the associated utility

        """

        # a. grid for x1 and the implied x2 (budget binds)
        x1_grid = np.linspace(0,self.I/self.p1,N)
        x2_grid = (self.I-self.p1*x1_grid)/self.p2

        # b. utility in all grid points at once
        u_grid = self.u_func(x1_grid,x2_grid)

        # c. the best point
        j = np.argmax(u_grid)

        return x1_grid[j],x2_grid[j],u_grid[j]

cp = ConsumerProblem()
x1_a,x2_a = cp.solve_analytical()
print(f'analytical: x1 = {x1_a:.6f}, x2 = {x2_a:.6f}, u = {cp.u_func(x1_a,x2_a):.6f}')
print('')

for N in [11,101,1001]:
    x1,x2,u = cp.solve(N)
    print(f'N = {N:5d}: x1 = {x1:.6f}, x2 = {x2:.6f}, u = {u:.6f}, error in x1 = {abs(x1-x1_a):.2e}')

analytical: x1 = 5.000000, x2 = 2.500000, u = 3.535534

N =    11: x1 = 5.000000, x2 = 2.500000, u = 3.535534, error in x1 = 0.00e+00
N =   101: x1 = 5.000000, x2 = 2.500000, u = 3.535534, error in x1 = 0.00e+00
N =  1001: x1 = 5.000000, x2 = 2.500000, u = 3.535534, error in x1 = 0.00e+00


**Check:** the numerical solution must (i) satisfy the budget constraint exactly, and (ii) never give higher utility than the analytical solution - it is a maximum over a *subset* of the feasible bundles.

In [37]:
x1,x2,u = cp.solve(1001)

print('expenditure =',cp.p1*x1+cp.p2*x2,'and I =',cp.I)
assert np.isclose(cp.p1*x1+cp.p2*x2, cp.I)

print('u (grid) =',u,'<= u (analytical) =',cp.u_func(x1_a,x2_a))
assert u <= cp.u_func(x1_a,x2_a) + 1e-12
print('checks passed')

expenditure = 10.0 and I = 10.0
u (grid) = 3.535533905932738 <= u (analytical) = 3.535533905932738
checks passed


**Note:** the grid solution is only as good as the grid. With an odd number of points spanning $[0,10]$ the analytical optimum $x_1^{\ast}=5$ happens to be a grid point here, so the error is zero - try `N = 10` or `N = 100` and see the error appear. A grid search is a blunt instrument; in lecture 5 we replace it with proper optimization algorithms.

## 12. <a id='toc12_'></a>[Summary](#toc0_)

**Summary of what you practiced:**

1. Writing a class with attributes and methods
2. Storing results of a `solve` method as attributes, and `__str__`
3. Flexible parameters with `**kwargs` and `setattr`
4. Operator methods (`__add__`, `__mul__`)
5. Creating arrays and inspecting `shape`/`ndim`/`dtype`
6. Views vs. copies
7. `*` vs. `@`
8. Broadcasting with `None`
9. Logical indexing with `&`, `|` and `~`
10. Reductions along an `axis`
11. Combining classes and numpy to solve an economic model - and checking against a known answer